# 03 - Customer Segmentation (K-Means)

**OmniStyle Customer Intelligence Platform**

This notebook builds an RFM-based K-Means segmentation of OmniStyle
customers, walking through feature scaling, k-selection (elbow +
silhouette), clustering, and business-friendly segment labeling.

This mirrors the logic in `src/train_model.py::run_segmentation`, but is
laid out step-by-step for exploration purposes. Running
`python -m src.train_model` will regenerate the authoritative
`data/processed/customer_segments.csv` used elsewhere in the project.


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from src.config import DATA_PROCESSED_DIR, RANDOM_SEED

np.random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
rfm = pd.read_csv(DATA_PROCESSED_DIR / "rfm_features.csv")
features = ["recency_days", "frequency", "monetary"]
X = rfm[features].fillna(0)
X.describe()


## Scale features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]


## Elbow method and silhouette analysis

In [ ]:
k_range = range(2, 9)
inertias, silhouettes = {}, {}

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias[k] = km.inertia_
    sample_idx = np.random.choice(len(X_scaled), min(2000, len(X_scaled)), replace=False)
    silhouettes[k] = silhouette_score(X_scaled[sample_idx], labels[sample_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(inertias.keys()), list(inertias.values()), marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(silhouettes.keys()), list(silhouettes.values()), marker="o", color="orange")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

silhouettes


**Note:** purely maximizing silhouette score on this dataset tends to favor
`k=2` (a stark active-vs-inactive split), which is statistically clean but
not very actionable for a marketing team. In `src/train_model.py`, we
constrain the search to `k` in `[4, 7]` so the resulting segments map onto
standard, actionable RFM personas. We do the same here for consistency.

In [ ]:
best_k = max({k: v for k, v in silhouettes.items() if 4 <= k <= 7}, key=silhouettes.get)
print("Selected k:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init=10)
rfm["cluster"] = kmeans.fit_predict(X_scaled)
rfm["cluster"].value_counts().sort_index()


## Label clusters using relative RFM ranking

In [ ]:
cluster_profile = rfm.groupby("cluster")[features].mean()
cluster_profile


In [ ]:
import sys
sys.path.append("..")
from src.train_model import label_segments

segment_labels = label_segments(cluster_profile)
rfm["segment"] = rfm["cluster"].map(segment_labels)
segment_labels


## Visualize segments

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
rfm["segment"].value_counts().plot(kind="barh", ax=ax, color="teal")
ax.set_title("Customer Segment Sizes")
ax.set_xlabel("Number of Customers")
plt.tight_layout()
plt.show()


In [ ]:
profile_by_segment = rfm.groupby("segment")[features].mean()
profile_norm = (profile_by_segment - profile_by_segment.min()) / (profile_by_segment.max() - profile_by_segment.min() + 1e-9)

profile_norm.plot(kind="bar", figsize=(9, 5))
plt.title("Normalized RFM Characteristics by Segment")
plt.ylabel("Normalized value (0-1)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Key takeaways

- Segment labels are derived from each cluster's *relative* recency,
  frequency, and monetary ranking versus the other clusters, not from
  arbitrary cluster IDs — this keeps the segmentation meaningful even as the
  underlying data changes.
- The resulting segments align with the archetype mix used to generate the
  synthetic data (highly active, loyal, new, at-risk, inactive, churned),
  which is a good sanity check that the pipeline is behaving as expected.
- These segments (`data/processed/customer_segments.csv`) feed directly into
  the Power BI "Customer Intelligence" dashboard page and the business
  recommendations report.
